# TruthGuard — Notebook 5/5
## Étape 5 : Évaluation avancée, Explicabilité SHAP, Cross-domain & Sauvegarde du pipeline

> **Prérequis** : Tous les modèles entraînés (Notebook 4), `results_df`, `ALL_PROBAS`, `tg_meta`, `test_meta` disponibles.

## 20. Courbes ROC & Precision-Recall

In [ ]:
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
palette_roc = ["#3498db","#e74c3c","#2ecc71","#f39c12","#9b59b6","#1abc9c"]

for idx, (name, proba) in enumerate(ALL_PROBAS.items()):
    color = palette_roc[idx % len(palette_roc)]

    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, lw=2, color=color,
                 label=f"{name} (AUC = {roc_auc:.3f})")

    prec, rec, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    axes[1].plot(rec, prec, lw=2, color=color,
                 label=f"{name} (AP = {ap:.3f})")

axes[0].plot([0,1],[0,1],"k--",lw=1.2,label="Aléatoire")
axes[0].fill_between([0,1],[0,1], alpha=0.05, color="gray")
axes[0].set_title("Courbes ROC", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Taux de Faux Positifs (FPR)")
axes[0].set_ylabel("Taux de Vrais Positifs (TPR)")
axes[0].legend(loc="lower right", fontsize=8.5)
axes[0].grid(alpha=0.4)

baseline_pr = y_test.mean()
axes[1].axhline(y=baseline_pr, color="gray", linestyle="--", lw=1.2,
                label=f"Baseline ({baseline_pr:.3f})")
axes[1].set_title("Courbes Precision-Recall", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend(loc="lower left", fontsize=8.5)
axes[1].grid(alpha=0.4)

plt.suptitle("Évaluation des courbes", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

## 21. Analyse des erreurs (Faux Positifs & Faux Négatifs)

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

y_pred_tg = tg_meta.predict(test_meta)

if "split" in df.columns:
    df_test_full = df[df["split"] == "test"].copy().reset_index(drop=True)
else:
    df_test_full = df.iloc[X_test_text.index].copy().reset_index(drop=True)

df_test_full["y_true"]  = y_test.values
df_test_full["y_pred"]  = y_pred_tg
df_test_full["y_proba"] = y_proba_tg

FP = df_test_full[(df_test_full["y_true"]==0) & (df_test_full["y_pred"]==1)]
FN = df_test_full[(df_test_full["y_true"]==1) & (df_test_full["y_pred"]==0)]
TP = df_test_full[(df_test_full["y_true"]==1) & (df_test_full["y_pred"]==1)]
TN = df_test_full[(df_test_full["y_true"]==0) & (df_test_full["y_pred"]==0)]

print("═" * 50)
print(f" Vrais Positifs  (TP) : {len(TP):>5}")
print(f" Vrais Négatifs  (TN) : {len(TN):>5}")
print(f" Faux Positifs   (FP) : {len(FP):>5}  ← Fake classifié comme Real")
print(f" Faux Négatifs   (FN) : {len(FN):>5}  ← Real classifié comme Fake")
print("═" * 50)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

groups = {
    "TP (Correct Real)": TP["clean_word_count"] if "clean_word_count" in TP.columns else TP["clean"].str.split().str.len(),
    "FP (Fake→Real)": FP["clean_word_count"] if "clean_word_count" in FP.columns else FP["clean"].str.split().str.len(),
    "FN (Real→Fake)": FN["clean_word_count"] if "clean_word_count" in FN.columns else FN["clean"].str.split().str.len(),
}
colors_g = ["#2980b9", "#e74c3c", "#f39c12"]

for ax, (name, data), color in zip(axes, groups.items(), colors_g):
    ax.hist(data.clip(upper=1000), bins=30, color=color, alpha=0.85)
    ax.axvline(data.median(), color="black", lw=2, linestyle="--",
               label=f"Médiane={data.median():.0f}")
    ax.set_title(name, fontweight="bold")
    ax.set_xlabel("Nombre de mots")
    ax.legend()

plt.suptitle("Longueur des textes : Erreurs vs Corrects", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("\n Top 3 Faux Positifs les plus confiants (Fake → classifié Real) :")
for _, row in FP.nlargest(3, "y_proba").iterrows():
    print(f"  [conf={row['y_proba']:.3f}] {str(row.get('statement', row.get('clean','')))[:120]}...")

print("\n Top 3 Faux Négatifs les plus confiants (Real → classifié Fake) :")
for _, row in FN.nsmallest(3, "y_proba").iterrows():
    print(f"  [conf={1-row['y_proba']:.3f}] {str(row.get('statement', row.get('clean','')))[:120]}...")

In [ ]:
thresholds = np.linspace(0.1, 0.9, 50)
f1s = [f1_score(df_test_full["y_true"], (df_test_full["y_proba"] >= t).astype(int))
       for t in thresholds]
precs = [precision_score(df_test_full["y_true"], (df_test_full["y_proba"] >= t).astype(int), zero_division=0)
         for t in thresholds]
recs = [recall_score(df_test_full["y_true"], (df_test_full["y_proba"] >= t).astype(int), zero_division=0)
        for t in thresholds]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for lbl, color, name in [(0, "#e74c3c", "Fake"), (1, "#2980b9", "Real")]:
    mask = df_test_full["y_true"] == lbl
    axes[0].hist(df_test_full[mask]["y_proba"], bins=40,
                 alpha=0.6, color=color, label=name, density=True)
axes[0].axvline(x=0.5, color="black", linestyle="--", lw=1.5, label="Seuil 0.5")
axes[0].set_title("Distribution des probabilités prédites", fontweight="bold")
axes[0].set_xlabel("P(Real)")
axes[0].legend()

best_t = thresholds[np.argmax(f1s)]
axes[1].plot(thresholds, f1s,   color="#9b59b6", lw=2, label="F1")
axes[1].plot(thresholds, precs, color="#3498db", lw=2, label="Precision")
axes[1].plot(thresholds, recs,  color="#e74c3c", lw=2, label="Recall")
axes[1].axvline(x=best_t, color="black", linestyle="--", lw=1.5,
                label=f"Meilleur seuil F1 = {best_t:.2f}")
axes[1].set_title("Métriques selon le seuil de décision", fontweight="bold")
axes[1].set_xlabel("Seuil")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()
print(f" Meilleur seuil F1 : {best_t:.2f}  (F1 = {max(f1s):.4f})")

## 22. Explicabilité SHAP

In [ ]:
try:
    import shap
    from sklearn.linear_model import LogisticRegression

    lr_explain = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
    lr_explain.fit(X_train_word, y_train)

    explainer   = shap.LinearExplainer(lr_explain, X_train_word,
                                        feature_perturbation="interventional")
    N_EXPLAIN   = 300
    shap_values = explainer.shap_values(X_test_word[:N_EXPLAIN])
    feature_names = word_tfidf.get_feature_names_out()

    plt.figure(figsize=(10, 7))
    shap.summary_plot(
        shap_values, X_test_word[:N_EXPLAIN].toarray(),
        feature_names=feature_names,
        max_display=20, show=False
    )
    plt.title("SHAP — Top 20 features les plus influentes (LR + TF-IDF)",
              fontweight="bold")
    plt.tight_layout(); plt.show()

    plt.figure(figsize=(9, 5))
    shap.summary_plot(
        shap_values, X_test_word[:N_EXPLAIN].toarray(),
        feature_names=feature_names,
        max_display=20, plot_type="bar", show=False
    )
    plt.title("SHAP — Importance globale des features", fontweight="bold")
    plt.tight_layout(); plt.show()

except ImportError:
    print(" SHAP non installé : pip install shap --break-system-packages")

## 23. Évaluation cross-domain

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

def cross_domain_eval(df_full, train_source, test_source):
    col = "clean" if "clean" in df_full.columns else "statement"
    df_tr = df_full[df_full["source_df"] == train_source].copy()
    df_te = df_full[df_full["source_df"] == test_source].copy()

    if df_tr.empty or df_te.empty:
        return None

    vec = TfidfVectorizer(max_features=10_000, ngram_range=(1,2), sublinear_tf=True)
    X_tr = vec.fit_transform(df_tr[col].fillna(""))
    X_te = vec.transform(df_te[col].fillna(""))

    clf = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
    clf.fit(X_tr, df_tr["label"])
    y_pred = clf.predict(X_te)

    return {
        "Train":     train_source,
        "Test":      test_source,
        "F1":        f1_score(df_te["label"], y_pred, zero_division=0),
        "Accuracy":  accuracy_score(df_te["label"], y_pred),
        "AUC":       roc_auc_score(df_te["label"], clf.predict_proba(X_te)[:,1]),
    }

sources = df["source_df"].unique().tolist()
cross_results = [
    cross_domain_eval(df, src_tr, src_te)
    for src_tr in sources for src_te in sources if src_tr != src_te
]
cross_results = [r for r in cross_results if r is not None]
cross_df = pd.DataFrame(cross_results)

print(cross_df.round(4).to_string(index=False))

In [ ]:
if not cross_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, metric in zip(axes, ["F1", "AUC"]):
        pivot = cross_df.pivot(index="Train", columns="Test", values=metric)
        sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGnBu",
                    vmin=0.3, vmax=1.0, linewidths=0.5, ax=ax)
        ax.set_title(f"Cross-domain {metric} (Train → Test)", fontweight="bold")

    plt.suptitle("Généralisation cross-domaine", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

    print(f"\n F1 moyen cross-domain : {cross_df['F1'].mean():.3f}")
    print(f"  F1 min  cross-domain  : {cross_df['F1'].min():.3f}")
    if cross_df['F1'].min() < 0.6:
        print("    Domain shift significatif — envisager du domain adaptation.")
    else:
        print("   Bonne généralisation cross-domain.")

## 24. Calibration des probabilités

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.model_selection import cross_val_score, StratifiedKFold

fig, axes = plt.subplots(1, len(ALL_PROBAS), figsize=(5 * len(ALL_PROBAS), 4))
if len(ALL_PROBAS) == 1:
    axes = [axes]

for ax, (name, proba) in zip(axes, ALL_PROBAS.items()):
    frac_pos, mean_pred = calibration_curve(y_test, proba, n_bins=10, strategy="uniform")
    ax.plot(mean_pred, frac_pos, "s-", color="#9b59b6", lw=2, label="Modèle")
    ax.plot([0,1],[0,1], "k--", lw=1.5, label="Calibration parfaite")
    ax.fill_between([0,1],[0,1], alpha=0.07, color="gray")
    ax.set_title(name, fontsize=9, fontweight="bold")
    ax.set_xlabel("Probabilité moyenne prédite")
    ax.set_ylabel("Fraction de positifs")
    ax.legend(fontsize=8)
    ax.set_xlim(0,1); ax.set_ylim(0,1)

plt.suptitle("Calibration des probabilités (Reliability Diagrams)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression

print(" Cross-validation 5-fold sur Logistic Regression (TF-IDF + Embeddings)...")
lr_cv = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
skf   = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

cv_f1  = cross_val_score(lr_cv, X_train_full, y_train, cv=skf, scoring="f1",       n_jobs=-1)
cv_auc = cross_val_score(lr_cv, X_train_full, y_train, cv=skf, scoring="roc_auc",  n_jobs=-1)

print(f"\n  F1  cross-val  : {cv_f1.mean():.4f} ± {cv_f1.std():.4f}")
print(f"  AUC cross-val  : {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, scores, title in zip(axes, [cv_f1, cv_auc], ["F1 par fold", "AUC-ROC par fold"]):
    ax.bar(range(1, 6), scores, color="#3498db", alpha=0.85, edgecolor="white")
    ax.axhline(scores.mean(), color="red", lw=2, linestyle="--",
               label=f"Moyenne = {scores.mean():.4f}")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Fold")
    ax.set_ylim(scores.min() * 0.97, 1.01)
    ax.legend()

plt.suptitle("Cross-validation 5-fold", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 25. Sauvegarde du pipeline & prédiction libre

In [ ]:
import joblib
import os

CACHE_DIR = Path("/content/drive/MyDrive/cache_data")
os.makedirs(CACHE_DIR, exist_ok=True)

pipeline_path = CACHE_DIR / "truthguard_pipeline_v2.pkl"

joblib.dump({
    "word_vectorizer":  word_tfidf,
    "char_vectorizer":  char_tfidf,
    "base_models":      [base_lr, base_rf, base_svm],
    "base_names":       ["LR", "RF", "SVM"],
    "meta_model":       tg_meta,
    "best_threshold":   best_t,
    "results_summary":  results_df.to_dict(),
}, pipeline_path)

print(f" Pipeline sauvegardé : {pipeline_path}")

In [ ]:
import re
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
import nltk
from sentence_transformers import SentenceTransformer
from scipy.sparse import hstack, csr_matrix
import numpy as np

nltk.download('averaged_perceptron_tagger_eng', quiet=True)

_STOP_WORDS = set(stopwords.words("english"))
_LEMMA      = WordNetLemmatizer()
model_st    = SentenceTransformer("all-MiniLM-L6-v2")

def _get_wn_pos(tag):
    return (wordnet.ADJ if tag.startswith('J') else
            wordnet.VERB if tag.startswith('V') else
            wordnet.ADV  if tag.startswith('R') else wordnet.NOUN)

def _clean(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in _STOP_WORDS and len(t) > 2]
    tokens = [_LEMMA.lemmatize(w, _get_wn_pos(t)) for w, t in pos_tag(tokens)]
    return " ".join(tokens)

def predict_news(text: str, threshold: float = None) -> dict:
    """Prédit si un texte est une fake news avec niveau de confiance."""
    threshold = threshold or best_t
    cleaned   = _clean(text)

    w_vec = word_tfidf.transform([cleaned])
    c_vec = char_tfidf.transform([cleaned])
    tfidf_combined_vec = hstack([w_vec, c_vec])
    text_embedding = model_st.encode([text])
    full_feature_vec = hstack([tfidf_combined_vec, csr_matrix(text_embedding)])

    meta = np.array([[m.predict_proba(full_feature_vec)[0, 1]
                      for m in [base_lr, base_rf, base_svm]]])
    proba = tg_meta.predict_proba(meta)[0, 1]
    label = " RÉEL" if proba >= threshold else " FAKE"

    margin = abs(proba - threshold)
    if margin < 0.1:   confidence_level = "Incertain"
    elif margin < 0.3: confidence_level = "Modéré"
    else:              confidence_level = "Élevé"

    return {
        "label":            label,
        "proba_real":       round(proba, 4),
        "proba_fake":       round(1 - proba, 4),
        "confidence":       f"{max(proba, 1-proba)*100:.1f}%",
        "confidence_level": confidence_level,
        "seuil_utilise":    threshold,
    }


samples = [
    "Scientists confirm new vaccine shows 95% efficacy in clinical trials conducted across 10 countries.",
    "Breaking: Government secretly implanting microchips through COVID vaccines to control citizens!",
    "Federal Reserve raises interest rates by 25 basis points amid inflation concerns.",
    "SHOCKING: Celebrity admits to being part of global satanic conspiracy controlling world governments.",
]

print("═" * 70)
print(" DÉMONSTRATION — Prédiction en temps réel")
print("═" * 70)
for s in samples:
    result = predict_news(s)
    print(f"\n {s[:80]}...")
    print(f"   → {result['label']}")
    print(f"   P(Real)={result['proba_real']:.3f} | P(Fake)={result['proba_fake']:.3f}"
          f" | Confiance: {result['confidence']} ({result['confidence_level']})")
    print("─" * 70)

In [ ]:
print("═" * 65)
print(" RÉCAPITULATIF FINAL")
print("═" * 65)

best_row = results_df[results_df["Features"]=="TF-IDF + Embeddings"].loc[
    results_df[results_df["Features"]=="TF-IDF + Embeddings"]["F1"].idxmax()
]
print(f"\n   Meilleur modèle : {best_row['Model']}")
print(f"     Accuracy  : {best_row['Accuracy']:.4f}")
print(f"     F1-Score  : {best_row['F1']:.4f}")
print(f"     AUC-ROC   : {best_row['AUC-ROC']:.4f}")
print(f"     AUC-PR    : {best_row['AUC-PR']:.4f}")
print(f"\n   Meilleur seuil de décision : {best_t:.2f}")
print(f"   Pipeline sauvegardé        : {pipeline_path}")
print("="*65)